# Pipeline Chatbot Bancario — Diagnóstico Proactivo

**Flujo:**
1. Usuario manda prompt (con `user_id` embebido)
2. Se codifica el prompt con el mismo modelo de embeddings
3. Se compara contra centroides de cada perfil temático
4. Se busca el historial del usuario en `features_conv.parquet`
5. Se arma un contexto rico y se manda a Qwen (Ollama)
6. El LLM propone soluciones proactivas

> **Prerequisito:** Ollama corriendo localmente con Qwen cargado.
> ```bash
> ollama serve
> ollama run qwen2.5:14b   # o el tag exacto que tengas
> ```

## 0. Imports y configuración

In [13]:
import numpy as np
import pandas as pd
import re
import requests
import json
from sentence_transformers import SentenceTransformer
import torch

# ── Rutas ──────────────────────────────────────────────────────────────
EMBEDDINGS_PATH   = "embeddings_conv.npy"              # todos los embeddings
CONV_FULL_PATH    = "conv_full_perfiles_finales.parquet" # conv_id, topic, perfil_final
FEATURES_PATH     = "features_conv.parquet"            # una fila por user_id
MAPEO_PATH        = "mapeo_topics_a_perfiles.csv"       # topic_id → palabras

# ── Ollama ─────────────────────────────────────────────────────────────
OLLAMA_URL        = "http://localhost:11434/api/generate"
OLLAMA_MODEL      = "qwen2.5:14b"   # ajusta al tag exacto que tengas

# ── Embedding model ────────────────────────────────────────────────────
EMBED_MODEL_NAME  = "paraphrase-multilingual-MiniLM-L12-v2"
TOP_K_PERFILES    = 2    # cuántos perfiles temáticos considerar
SIM_THRESHOLD     = 0.30 # similitud mínima para considerar un perfil relevante

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

Device: cuda


## 1. Cargar datos y calcular centroides por perfil

In [14]:
print("Cargando embeddings y metadata...")
embeddings  = np.load(EMBEDDINGS_PATH)           # shape (N, 384)
conv_full   = pd.read_parquet(CONV_FULL_PATH)    # N filas
features    = pd.read_parquet(FEATURES_PATH)     # una fila por usuario
mapeo       = pd.read_csv(MAPEO_PATH)            # topic_id, palabras, perfil_macro

print(f"  Embeddings : {embeddings.shape}")
print(f"  conv_full  : {conv_full.shape}")
print(f"  features   : {features.shape}")
print(f"  Perfiles únicos: {conv_full['perfil_final'].nunique()}")

Cargando embeddings y metadata...
  Embeddings : (23791, 384)
  conv_full  : (23791, 10)
  features   : (14928, 32)
  Perfiles únicos: 27


In [15]:
# ── Calcular centroide (media) de embeddings por perfil_final ──────────
# Aseguramos que conv_full e índice de embeddings estén alineados
assert len(embeddings) == len(conv_full), \
    "Mismatch entre embeddings y conv_full — revisa que no hayas filtrado filas después de generar los embeddings"

conv_full = conv_full.reset_index(drop=True)
conv_full["_emb_idx"] = conv_full.index   # índice seguro

centroides = {}
for perfil, grupo in conv_full.groupby("perfil_final"):
    if perfil in ("Sin clasificar", "Otros"):
        continue
    idxs = grupo["_emb_idx"].values
    vecs = embeddings[idxs]          # (k, 384)
    c = vecs.mean(axis=0)
    c = c / (np.linalg.norm(c) + 1e-10)   # normalizar
    centroides[perfil] = c

# Matriz de centroides para búsqueda vectorizada
perfil_labels  = list(centroides.keys())
centroid_matrix = np.stack([centroides[p] for p in perfil_labels])  # (P, 384)

print(f"Centroides calculados: {len(perfil_labels)} perfiles")
print("Perfiles:", perfil_labels[:10], "...")

Centroides calculados: 25 perfiles
Perfiles: ['Atención humana', 'Autenticación / token', 'CLABE / datos bancarios', 'Configuración / contrato', 'Depósitos en efectivo', 'Disputas y problemas con cargos', 'Eliminar / dar de baja', 'Gestión de tarjeta', 'Hey Pro y beneficios', 'Inversión'] ...


## 2. Cargar modelo de embeddings

In [16]:
print("Cargando modelo de embeddings...")
embed_model = SentenceTransformer(EMBED_MODEL_NAME, device=device)
print("Modelo listo.")

Cargando modelo de embeddings...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Modelo listo.


## 3. Funciones del pipeline

In [17]:
# ── 3.1  Limpiar texto (igual que en el notebook de clustering) ────────
def limpiar(t: str) -> str:
    t = str(t).lower()
    t = re.sub(r"\s+", " ", t)
    t = re.sub(r"http\S+", "", t)
    return t.strip()


# ── 3.2  Clasificar prompt contra centroides ───────────────────────────
def clasificar_prompt(prompt: str, top_k: int = TOP_K_PERFILES,
                      threshold: float = SIM_THRESHOLD):
    """
    Devuelve lista de (perfil, similitud_coseno) ordenada de mayor a menor.
    Solo incluye perfiles por encima del threshold.
    """
    texto_limpio = limpiar(prompt)
    vec = embed_model.encode(
        [texto_limpio],
        normalize_embeddings=True,
        convert_to_numpy=True
    )[0]  # (384,)

    sims = centroid_matrix @ vec          # (P,) producto punto = coseno (ya normalizados)
    top_idx = np.argsort(sims)[::-1][:top_k]

    resultados = [
        (perfil_labels[i], float(sims[i]))
        for i in top_idx
        if float(sims[i]) >= threshold
    ]
    return resultados


# ── 3.3  Buscar perfil del usuario en features_conv ───────────────────
def buscar_usuario(user_id: str) -> dict | None:
    """
    Devuelve un dict con el perfil dominante del usuario y los
    3 perfiles con mayor porcentaje de conversaciones.
    Retorna None si el user_id no existe.
    """
    fila = features[features["user_id"].astype(str) == str(user_id)]
    if fila.empty:
        return None

    row = fila.iloc[0]
    perfil_dom = row.get("perfil_dominante", "desconocido")
    n_conv     = int(row.get("n_conversaciones", 0))
    n_perfiles = int(row.get("n_perfiles_tocados", 0))

    # Top-3 perfiles por porcentaje
    pct_cols = [c for c in features.columns if c.startswith("pct_")]
    top_perfiles = (
        row[pct_cols]
        .sort_values(ascending=False)
        .head(3)
    )
    top_perfiles_dict = {
        c.replace("pct_", "").replace("_", " "): round(float(v), 3)
        for c, v in top_perfiles.items()
    }

    return {
        "user_id":         str(user_id),
        "perfil_dominante": perfil_dom,
        "n_conversaciones": n_conv,
        "n_perfiles_tocados": n_perfiles,
        "top_perfiles_pct": top_perfiles_dict,
    }


# ── 3.4  Extraer user_id del prompt ───────────────────────────────────
def extraer_user_id(prompt: str) -> tuple[str | None, str]:
    patron = r"user[_\-]?id\s*=\s*([\w\d]+)"
    m = re.search(patron, prompt, re.IGNORECASE)
    if m:
        uid = m.group(1)
        prompt_limpio = re.sub(patron, "", prompt, flags=re.IGNORECASE).strip()
        return uid, prompt_limpio
    return None, prompt


print("Funciones del pipeline definidas.")

Funciones del pipeline definidas.


In [18]:
# ── 3.5  Armar el system prompt para Qwen ─────────────────────────────
SYSTEM_PROMPT = """Eres un asistente virtual de un banco. Tu objetivo es ayudar proactivamente al usuario.

Se te proporcionará:
- El mensaje del usuario
- Los temas más probables relacionados con su mensaje (basados en análisis de conversaciones similares)
- El historial de interacciones previas del usuario con el banco

Tu tarea:
1. Identifica los 2 o 3 problemas más probables que el usuario podría estar enfrentando
2. Para cada problema, proporciona pasos concretos de solución
3. Sé directo, empático y claro
4. No menciones datos técnicos internos (embeddings, clusters, perfiles) en tu respuesta
5. Habla siempre en español, en tono profesional pero cercano
6. Si el usuario ya tiene historial con ese tema, reconócelo sutilmente

Siempre desarrolla cada solución con al menos 3-4 pasos detallados. No des respuestas cortas.
"""

def construir_prompt_llm(prompt_usuario: str,
                         perfiles_detectados: list,
                         datos_usuario: dict | None) -> str:
    """
    Arma el prompt completo que recibe el LLM con todo el contexto.
    """
    # Sección de temas detectados
    if perfiles_detectados:
        temas_str = "\n".join(
            f"  - {perfil} (relevancia: {sim:.2f})"
            for perfil, sim in perfiles_detectados
        )
    else:
        temas_str = "  - No se detectó un tema específico (consulta general)"

    # Sección de historial del usuario
    if datos_usuario:
        hist_str = (
            f"  - Perfil dominante: {datos_usuario['perfil_dominante']}\n"
            f"  - Total conversaciones previas: {datos_usuario['n_conversaciones']}\n"
            f"  - Temas que ha consultado antes: "
            + ", ".join(
                f"{k} ({v:.0%})"
                for k, v in datos_usuario["top_perfiles_pct"].items()
            )
        )
    else:
        hist_str = "  - Usuario nuevo, sin historial previo"

    prompt_completo = f"""=== MENSAJE DEL USUARIO ===
{prompt_usuario}

=== TEMAS PROBABLES (análisis interno) ===
{temas_str}

=== HISTORIAL DEL USUARIO ===
{hist_str}

=== INSTRUCCIÓN ===
Con base en el mensaje y el contexto anterior, propón de forma proactiva los 2 o 3 problemas más probables
que este usuario podría estar enfrentando y ofrece soluciones concretas para cada uno.
Responde directamente al usuario."""

    return prompt_completo


print("System prompt definido.")

System prompt definido.


In [19]:
# ── 3.6  Llamada a Ollama ──────────────────────────────────────────────
def llamar_ollama(prompt_llm: str,
                  system: str = SYSTEM_PROMPT,
                  model: str = OLLAMA_MODEL,
                  stream: bool = False) -> str:
    """
    Llama al endpoint /api/generate de Ollama.
    Retorna el texto generado por el modelo.
    """
    payload = {
        "model":  model,
        "prompt": prompt_llm,
        "system": system,
        "stream": stream,
        "options": {
            "temperature": 0.6,
            "top_p": 0.9,
            "num_predict": 2048,
            "num_ctx": 4096,
        }
    }
    try:
        resp = requests.post(OLLAMA_URL, json=payload, timeout=300)
        resp.raise_for_status()
        data = resp.json()
        return data.get("response", "[sin respuesta]").strip()
    except requests.exceptions.ConnectionError:
        return "[ERROR] No se pudo conectar con Ollama. ¿Está corriendo 'ollama serve'?"
    except Exception as e:
        return f"[ERROR] {str(e)}"


print("Función Ollama definida.")

Función Ollama definida.


## 4. Pipeline completo

In [20]:
def pipeline(prompt_raw: str, verbose: bool = True) -> str:
    """
    Pipeline completo: prompt crudo → respuesta del LLM.
    
    El prompt debe contener el user_id en alguno de estos formatos:
        user_id: 12345  |  [12345]  |  id=12345
    """
    sep = "─" * 55

    # 1. Extraer user_id
    user_id, prompt_usuario = extraer_user_id(prompt_raw)
    if verbose:
        print(sep)
        print(f"[1] user_id detectado : {user_id}")
        print(f"    prompt limpio     : {prompt_usuario[:120]}")

    # 2. Clasificar contra centroides
    perfiles_detectados = clasificar_prompt(prompt_usuario)
    if verbose:
        print(sep)
        print("[2] Perfiles detectados:")
        for p, s in perfiles_detectados:
            print(f"    {p:35s}  sim={s:.3f}")
        if not perfiles_detectados:
            print("    (ninguno por encima del threshold)")

    # 3. Buscar datos del usuario
    datos_usuario = buscar_usuario(user_id) if user_id else None
    if verbose:
        print(sep)
        print("[3] Datos del usuario:")
        if datos_usuario:
            print(f"    Perfil dominante   : {datos_usuario['perfil_dominante']}")
            print(f"    N conversaciones   : {datos_usuario['n_conversaciones']}")
            print(f"    Top perfiles       : {datos_usuario['top_perfiles_pct']}")
        else:
            print("    Usuario no encontrado en la base de datos")

    # 4. Armar contexto para el LLM
    prompt_llm = construir_prompt_llm(prompt_usuario, perfiles_detectados, datos_usuario)
    if verbose:
        print(sep)
        print("[4] Prompt enviado al LLM:")
        print(prompt_llm)

    # 5. Llamar a Ollama
    if verbose:
        print(sep)
        print(f"[5] Llamando a {OLLAMA_MODEL}...")
    respuesta = llamar_ollama(prompt_llm)

    if verbose:
        print(sep)
        print("[6] RESPUESTA DEL CHATBOT:")
        print(respuesta)
        print(sep)

    return respuesta


print("Pipeline listo.")

Pipeline listo.


## 5. Pruebas

In [21]:
# ── Prueba 1: usuario con historial ───────────────────────────────────
# Cambia el user_id por uno que exista en tu features_conv.parquet
USUARIO_PRUEBA = features["user_id"].iloc[0]   # toma el primero disponible

#prompt_prueba_1 = f"user_id: {USUARIO_PRUEBA} Hola, tengo un problema con mi tarjeta, no me deja hacer pagos"

#respuesta_1 = pipeline(prompt_prueba_1, verbose=True)

In [22]:
# ── Prueba 2: usuario desconocido ─────────────────────────────────────
#prompt_prueba_2 = "user_id: XXXXXXXX No puedo ver mi estado de cuenta del mes pasado"

#respuesta_2 = pipeline(prompt_prueba_2, verbose=True)

In [23]:
# ── Prueba 3: consulta ambigua ─────────────────────────────────────────
#prompt_prueba_3 = f"user_id: {USUARIO_PRUEBA} quiero saber cómo activar esto que me mandaron"

#respuesta_3 = pipeline(prompt_prueba_3, verbose=True)

## 6. Uso interactivo rápido

Corre esta celda para probar cualquier prompt de forma rápida.

In [24]:
# Edita este prompt y vuelve a correr la celda
MI_PROMPT = f"user_id: {USUARIO_PRUEBA} ¿Como puedo iniciar con mis inversiones en hey banco?"

_ = pipeline(MI_PROMPT, verbose=True)

───────────────────────────────────────────────────────
[1] user_id detectado : None
    prompt limpio     : user_id: USR-00001 ¿Como puedo iniciar con mis inversiones en hey banco?
───────────────────────────────────────────────────────
[2] Perfiles detectados:
    Problemas de app                     sim=0.578
    Depósitos en efectivo                sim=0.543
───────────────────────────────────────────────────────
[3] Datos del usuario:
    Usuario no encontrado en la base de datos
───────────────────────────────────────────────────────
[4] Prompt enviado al LLM:
=== MENSAJE DEL USUARIO ===
user_id: USR-00001 ¿Como puedo iniciar con mis inversiones en hey banco?

=== TEMAS PROBABLES (análisis interno) ===
  - Problemas de app (relevancia: 0.58)
  - Depósitos en efectivo (relevancia: 0.54)

=== HISTORIAL DEL USUARIO ===
  - Usuario nuevo, sin historial previo

=== INSTRUCCIÓN ===
Con base en el mensaje y el contexto anterior, propón de forma proactiva los 2 o 3 problemas más probable

---
## Notas de configuración

| Parámetro | Dónde cambiar | Descripción |
|---|---|---|
| `OLLAMA_MODEL` | Celda 0 | Tag exacto del modelo en Ollama |
| `TOP_K_PERFILES` | Celda 0 | Cuántos perfiles considerar (2 recomendado) |
| `SIM_THRESHOLD` | Celda 0 | Similitud mínima (baja a 0.20 si hay pocos hits) |
| `extraer_user_id` | Celda 3.4 | Ajusta el regex al formato real de tu sistema |
| `SYSTEM_PROMPT` | Celda 3.5 | Personaliza el tono y las instrucciones al LLM |
| `temperature` | Celda 3.6 | 0.3–0.5 para respuestas más consistentes |
